# Implementing Early-Exit on Resnet model

## What is Early Exit?
In recent researches, some models have been implemented that they have more than one output. Exept last one, the others are called Early-Exits. By using that, The inference time will decrease and less inference memory. In this project, I implemented EE on Resnet families. The implementation in compleatly flexible. What means that you can create EE with your costum structure and set it where do you exactly want.

### IMPORTANT
Only inference time and memory will reduce. The train time and validating time and their memory usage will defeneatly increase and the reason is related to training all neurons in the model. There for in trainging step, all data should be showen to all backbone and exits

### IMPORTANT
To exit earlier, the output should pass the treshold. Therefor, make sure the trshold has logical quantity and you have enougth exits. No more no less.

## Import libraries here
There is all libraries you need to implemet.

### NOTE
If you want to use M chipset Mack, you need to use `mps` to get access to your GPU.

In [ ]:
import numpy as np
import torch
import torch.backends
import torch.backends.mps
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler
from tqdm import tqdm
import time

## Optional block
In some cases, during implementation of CNN models, you need to implement pooling layer right after convolutional layer. Use this block to implement this situation.

In [ ]:
class ConvPoolAc(nn.Module):
    def __init__(self, chanOut, kernel=3, stride=1, padding=1, p_ceil_mode=False, pool_pad=0):
        super(ConvPoolAc, self).__init__()

        self.layer = nn.Sequential(
            nn.LazyConv2d(chanOut, kernel_size=kernel,
                stride=stride, padding=padding, bias=False),
            nn.AvgPool2d(2, stride=2, ceil_mode=p_ceil_mode, padding=pool_pad), #ksize, stride
            nn.ReLU(True)
        )

    def forward(self, x):
        return self.layer(x)

In [ ]:
class AdvancedConvPoolAc(nn.Module):
    def __init__(self, features):
        super(AdvancedConvPoolAc, self).__init__()
        self.layer = nn.ModuleList()
        for i, item in enumerate(features):
            if item == "conv":
                conv_features = features[i + 1]
                self.layer.append(nn.LazyConv2d(conv_features["channel"],
                                                kernel_size=conv_features["kernel"],
                                                stride=conv_features["stride"],
                                                padding=conv_features["padding"], bias=False))
                self.layer.append(nn.ReLU(True))
            elif item == "pool":
                pool_features = features[i + 1]
                self.layer.append(nn.AvgPool2d(pool_features["channel"],
                                               stride=pool_features["stride"],
                                               ceil_mode=False, padding=0))

    def forward(self, x):
        for layer in self.layer:
            x = layer(x)
        return x